# 🧠 GenAI Core Algorithms, Tools & Evaluation Masterclass
### *The Definitive Live-Coding & Systems Handbook for MAANG & Tier-1 AI Interviews*

---

## 🎯 What This Notebook Covers
Every top-tier Generative AI / ML Systems interview (Meta, Google, OpenAI, Anthropic, Apple, Microsoft) tests candidates on **first-principles implementations** of core mathematical algorithms without relying on high-level wrappers like LangChain or LlamaIndex.

This notebook builds **every essential GenAI tool, retrieval algorithm, Transformer mechanism, optimization layer, and evaluation metric from scratch** in pure Python & vectorized NumPy:

1. **Distance Metrics & Metric Spaces** (Cosine, Euclidean L2, Manhattan L1, Angular)
2. **Vector Search & Hybrid Retrieval** (BM25 Okapi, IVFFlat with K-Means, HNSW Graph Search, Reciprocal Rank Fusion)
3. **Production In-Memory Vector Database** (CRUD, Token Chunking, Metadata ACL Filtering, Top-$K$ Search)
4. **Core Transformer Mechanics** (Byte-Pair Encoding, Scaled Dot-Product MHA, Grouped-Query Attention with KV-Cache, RoPE)
5. **Quantization & PEFT** (Symmetric/Asymmetric INT8 Quantization, LoRA Layer)
6. **Production Agentic Tooling & System Infrastructure** (Semantic Model Router, Air-Gapped Tool Calling & RBAC Gatekeeper, Context Budget Compressor, PII Scrubbing)
7. **The Complete Evaluation Suite** (The RAG Triad: Faithfulness, Relevance; Perplexity, ROUGE-1/2/L, BLEU-4, Token F1, LLM-as-a-Judge)

---


In [ ]:
# Setup & Core Dependencies
import math
import re
import json
import time
import random
import heapq
from typing import List, Dict, Any, Tuple, Optional, Set
from collections import Counter, defaultdict
import numpy as np

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("🚀 Environment initialized successfully with NumPy & Standard Libraries!")


---
# 📐 Section 1: Distance Metrics & Metric Spaces (RAG Foundations)

In retrieval and embedding spaces, distance metrics compute semantic proximity between high-dimensional embeddings.

| Metric | Formula | Best Used For | Interview Trap |
| :--- | :--- | :--- | :--- |
| **Cosine Similarity** | $\frac{u \cdot v}{\|u\| \|v\|}$ | Directional semantic alignment regardless of vector magnitude | Zero-norm division ($\|u\| = 0$). Must add numerical stability $\epsilon = 1e-9$. |
| **Euclidean Distance (L2)** | $\sqrt{\sum (u_i - v_i)^2}$ | Spatial geometric distance | If embeddings are L2-normalized ($\|u\|=1$), $L_2^2 = 2 - 2 \cos(u, v)$. |
| **Manhattan Distance (L1)** | $\sum \|u_i - v_i\|$ | Robustness to extreme outlier dimensions | Sparse feature spaces. |
| **Angular Distance** | $\frac{\arccos(\text{Cosine})}{\pi}$ | True distance metric adhering to triangle inequality | Range $[0, 1]$, maps angle directly to a distance metric. |


In [ ]:
# 1.1 Vectorized Distance Metrics Implementation

def cosine_similarity(u: np.ndarray, v: np.ndarray, eps: float = 1e-9) -> float:
    """
    Computes Cosine Similarity between vectors u and v with epsilon stability.
    Range: [-1.0, 1.0] (1.0 = identical direction).
    """
    u = np.asarray(u, dtype=np.float32)
    v = np.asarray(v, dtype=np.float32)
    dot = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    return float(dot / (max(norm_u * norm_v, eps)))

def batch_cosine_similarity(query_vec: np.ndarray, doc_matrix: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    """
    Computes Cosine Similarity between a query (1, D) and N document vectors (N, D).
    Uses vectorized matrix multiplication without Python loops.
    """
    query_norm = query_vec / max(np.linalg.norm(query_vec), eps)
    doc_norms = doc_matrix / np.maximum(np.linalg.norm(doc_matrix, axis=1, keepdims=True), eps)
    return np.dot(doc_norms, query_norm)

def euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    """Computes Euclidean (L2) distance."""
    return float(np.sqrt(np.sum((np.asarray(u) - np.asarray(v)) ** 2)))

def manhattan_distance(u: np.ndarray, v: np.ndarray) -> float:
    """Computes Manhattan (L1) distance."""
    return float(np.sum(np.abs(np.asarray(u) - np.asarray(v))))

def angular_distance(u: np.ndarray, v: np.ndarray) -> float:
    """Computes normalized Angular Distance in range [0, 1]."""
    cos_sim = np.clip(cosine_similarity(u, v), -1.0, 1.0)
    return float(np.arccos(cos_sim) / np.pi)

# Unit Test & Verification
vec_a = np.array([1.0, 2.0, 3.0])
vec_b = np.array([2.0, 4.0, 6.0]) # Collinear with vec_a
vec_c = np.array([-1.0, -2.0, -3.0]) # Opposite

print(f"Cosine Sim (Identical Direction): {cosine_similarity(vec_a, vec_b):.4f}") # 1.0000
print(f"Cosine Sim (Opposite Direction):  {cosine_similarity(vec_a, vec_c):.4f}") # -1.0000
print(f"Euclidean Distance (a to b):      {euclidean_distance(vec_a, vec_b):.4f}") # 3.7417
print(f"Angular Distance (a to c):        {angular_distance(vec_a, vec_c):.4f}") # 1.0000

# Batch test
docs = np.random.randn(100, 128)
query = np.random.randn(128)
batch_scores = batch_cosine_similarity(query, docs)
assert batch_scores.shape == (100,)
print(f"✅ Batch Cosine Similarity verified over 100 documents! Top score: {np.max(batch_scores):.4f}")


---
# 🔍 Section 2: Vector Search & Indexing Algorithms (RAG)

In enterprise RAG, brute-force linear search $\mathcal{O}(N \cdot D)$ is too slow when $N > 1,000,000$. Search systems utilize **Approximate Nearest Neighbor (ANN)** indexing (IVFFlat, HNSW) combined with **sparse lexical retrieval** (BM25) fused via **Reciprocal Rank Fusion (RRF)**.

```mermaid
flowchart LR
    A[User Query] --> B1[Dense Vector Embeddings]
    A --> B2[Sparse BM25 Tokens]
    B1 --> C1[IVFFlat / HNSW Search]
    B2 --> C2[Inverted Index Search]
    C1 --> D[Reciprocal Rank Fusion RRF]
    C2 --> D
    D --> E[Top-K Fused Candidates]
```


In [ ]:
# 2.1 BM25 Okapi Sparse Keyword Retrieval Engine

class BM25Okapi:
    """
    BM25 Okapi retrieval from scratch.
    Score formula:
    IDF(q_i) * (TF * (k1 + 1)) / (TF + k1 * (1 - b + b * (doc_len / avg_doc_len)))
    """
    def __init__(self, corpus: List[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_lengths = []
        self.doc_token_counts = []
        self.doc_freqs = defaultdict(int)
        self.idf = {}
        
        # Tokenize & build index
        self._build_index(corpus)
        
    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\w+', text.lower())
        
    def _build_index(self, corpus: List[str]):
        for doc in corpus:
            tokens = self._tokenize(doc)
            self.doc_lengths.append(len(tokens))
            counts = Counter(tokens)
            self.doc_token_counts.append(counts)
            for token in counts:
                self.doc_freqs[token] += 1
                
        self.avg_doc_len = sum(self.doc_lengths) / max(self.corpus_size, 1)
        
        # Calculate IDF with standard Okapi smoothing
        for term, freq in self.doc_freqs.items():
            self.idf[term] = math.log(1 + (self.corpus_size - freq + 0.5) / (freq + 0.5))
            
    def search(self, query: str, top_k: int = 3) -> List[Tuple[int, float]]:
        query_tokens = self._tokenize(query)
        scores = []
        
        for idx, counts in enumerate(self.doc_token_counts):
            score = 0.0
            doc_len = self.doc_lengths[idx]
            for token in query_tokens:
                if token in counts:
                    tf = counts[token]
                    idf = self.idf.get(token, 0.0)
                    numerator = tf * (self.k1 + 1.0)
                    denominator = tf + self.k1 * (1.0 - self.b + self.b * (doc_len / self.avg_doc_len))
                    score += idf * (numerator / denominator)
            scores.append((idx, score))
            
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

# Demo BM25
corpus = [
    "Transformer models utilize self-attention mechanisms for natural language processing.",
    "Reciprocal Rank Fusion fuses lexical BM25 search with dense vector similarity.",
    "Kubernetes manages containerized microservices and automated cluster scaling.",
    "KV-cache drastically reduces decoder Time-To-First-Token in large language models."
]

bm25 = BM25Okapi(corpus)
results = bm25.search("BM25 hybrid reciprocal rank search", top_k=2)
print("BM25 Search Results:")
for idx, score in results:
    print(f"  [Doc {idx}] Score: {score:.4f} -> '{corpus[idx]}'")


In [ ]:
# 2.2 IVFFlat (Inverted File Flat) Indexer with K-Means Clustering

class IVFFlatIndex:
    """
    Inverted File Index (IVF-Flat) partitioner.
    Divides high-dimensional space into n_clusters Voronoi cells using K-Means.
    Queries only probe the 'n_probe' nearest cluster centroids, cutting search from O(N) to O(n_probe * (N/C)).
    """
    def __init__(self, dim: int, n_clusters: int = 4, n_probe: int = 2):
        self.dim = dim
        self.n_clusters = n_clusters
        self.n_probe = n_probe
        self.centroids = None
        self.inverted_lists = defaultdict(list) # centroid_id -> list of (doc_id, vector)
        
    def fit_and_index(self, vectors: np.ndarray, doc_ids: List[str]):
        N, D = vectors.shape
        # Simple K-Means clustering for centroids
        indices = np.random.choice(N, self.n_clusters, replace=False)
        self.centroids = vectors[indices].copy()
        
        # 5 iterations of K-Means
        for _ in range(5):
            clusters = defaultdict(list)
            for vec in vectors:
                c_idx = np.argmin(np.linalg.norm(self.centroids - vec, axis=1))
                clusters[c_idx].append(vec)
            for c_idx in range(self.n_clusters):
                if clusters[c_idx]:
                    self.centroids[c_idx] = np.mean(clusters[c_idx], axis=0)
                    
        # Populate inverted lists
        for doc_id, vec in zip(doc_ids, vectors):
            c_idx = int(np.argmin(np.linalg.norm(self.centroids - vec, axis=1)))
            self.inverted_lists[c_idx].append((doc_id, vec))
            
    def search(self, query_vec: np.ndarray, top_k: int = 3) -> List[Tuple[str, float]]:
        # 1. Find the nearest n_probe centroids
        centroid_dists = np.linalg.norm(self.centroids - query_vec, axis=1)
        probed_centroids = np.argsort(centroid_dists)[:self.n_probe]
        
        candidates = []
        for c_idx in probed_centroids:
            for doc_id, vec in self.inverted_lists[c_idx]:
                sim = cosine_similarity(query_vec, vec)
                candidates.append((doc_id, sim))
                
        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[:top_k]

# Demo IVFFlat
data_vecs = np.random.randn(200, 16)
data_ids = [f"doc_{i}" for i in range(200)]
ivf = IVFFlatIndex(dim=16, n_clusters=8, n_probe=2)
ivf.fit_and_index(data_vecs, data_ids)

q = np.random.randn(16)
ivf_results = ivf.search(q, top_k=3)
print("IVFFlat Search Results:", ivf_results)


In [ ]:
# 2.3 Hierarchical Navigable Small World (HNSW) Graph Search Simulator

class HNSWLayer:
    """
    A single layer of an HNSW proximity graph.
    Uses Greedy routing to find the nearest neighbor to a query vector.
    """
    def __init__(self, M: int = 4):
        self.M = M # max connections per node
        self.nodes = {} # node_id -> vector
        self.adj = defaultdict(list) # node_id -> list of neighbor node_ids
        
    def add_node(self, node_id: str, vector: np.ndarray):
        self.nodes[node_id] = vector
        if len(self.nodes) > 1:
            # Connect to up to M closest existing nodes
            dists = [(other_id, euclidean_distance(vector, other_vec)) 
                     for other_id, other_vec in self.nodes.items() if other_id != node_id]
            dists.sort(key=lambda x: x[1])
            neighbors = [nid for nid, _ in dists[:self.M]]
            self.adj[node_id] = neighbors
            for nid in neighbors:
                if len(self.adj[nid]) < self.M:
                    self.adj[nid].append(node_id)
                    
    def greedy_search(self, query: np.ndarray, entry_point: str) -> Tuple[str, float]:
        curr = entry_point
        curr_dist = euclidean_distance(query, self.nodes[curr])
        
        while True:
            changed = False
            for neighbor in self.adj[curr]:
                dist = euclidean_distance(query, self.nodes[neighbor])
                if dist < curr_dist:
                    curr = neighbor
                    curr_dist = dist
                    changed = True
            if not changed:
                break
        return curr, curr_dist

# Demo HNSW layer
hnsw = HNSWLayer(M=3)
for i in range(10):
    hnsw.add_node(f"node_{i}", np.random.randn(8))

best_node, dist = hnsw.greedy_search(np.random.randn(8), entry_point="node_0")
print(f"HNSW Graph Routing Found Nearest: {best_node} with L2 dist {dist:.4f}")


In [ ]:
# 2.4 Reciprocal Rank Fusion (RRF) Hybrid Ranker

def reciprocal_rank_fusion(
    ranked_lists: List[List[str]], 
    k: int = 60,
    weights: Optional[List[float]] = None
) -> List[Tuple[str, float]]:
    """
    Reciprocal Rank Fusion (RRF) Formula:
    RRF(d) = sum_{m in M} ( w_m / (k + rank_m(d)) )
    
    Eliminates the need to normalize disparate score scales (e.g. BM25 unbounded vs Cosine [-1, 1]).
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
        
    scores = defaultdict(float)
    for list_idx, rank_list in enumerate(ranked_lists):
        w = weights[list_idx]
        for rank, doc_id in enumerate(rank_list):
            scores[doc_id] += w * (1.0 / (k + rank + 1))
            
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs

# Demo RRF
bm25_ranking = ["doc_A", "doc_B", "doc_C", "doc_D"]
dense_ranking = ["doc_C", "doc_A", "doc_E", "doc_B"]

fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking], k=60, weights=[0.4, 0.6])
print("Reciprocal Rank Fusion (RRF) Combined Rankings:")
for doc, score in fused:
    print(f"  {doc}: {score:.6f}")


---
# 🗄️ Section 3: Production In-Memory Vector Database with Metadata ACLs

Enterprise RAG demands strict **Multi-Tenant Document Access Control Lists (ACLs)** and metadata filtering so users never retrieve confidential documents outside their department.


In [ ]:
# 3.1 Complete MiniVectorDB with Metadata Filtering & Persistence

class MiniVectorDB:
    """
    A self-contained in-memory Vector Database supporting:
    - Document chunking & embedding storage
    - Pre-filtering by multi-tenant ACL clearance & department
    - Top-K cosine similarity semantic retrieval
    - JSON serialization / persistence
    """
    def __init__(self, embedding_dim: int = 128):
        self.embedding_dim = embedding_dim
        self.documents: Dict[str, Dict[str, Any]] = {}
        self.embeddings: Dict[str, np.ndarray] = {}
        
    def add_document(self, doc_id: str, text: str, embedding: np.ndarray, metadata: Dict[str, Any]):
        assert embedding.shape == (self.embedding_dim,), "Embedding dimension mismatch"
        self.documents[doc_id] = {
            "text": text,
            "metadata": metadata
        }
        self.embeddings[doc_id] = embedding / max(np.linalg.norm(embedding), 1e-9)
        
    def query(
        self, 
        query_embedding: np.ndarray, 
        top_k: int = 3,
        tenant_id: Optional[str] = None,
        required_role: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        query_norm = query_embedding / max(np.linalg.norm(query_embedding), 1e-9)
        candidates = []
        
        for doc_id, doc in self.documents.items():
            meta = doc["metadata"]
            # Tenant isolation filter
            if tenant_id and meta.get("tenant_id") != tenant_id:
                continue
            # Role clearance filter
            if required_role and required_role not in meta.get("clearance", []):
                continue
                
            sim = float(np.dot(self.embeddings[doc_id], query_norm))
            candidates.append({
                "doc_id": doc_id,
                "text": doc["text"],
                "score": round(sim, 4),
                "metadata": meta
            })
            
        candidates.sort(key=lambda x: x["score"], reverse=True)
        return candidates[:top_k]

# Demo Vector DB
vdb = MiniVectorDB(embedding_dim=4)
vdb.add_document("doc_1", "Turbine repair manual", np.array([0.9, 0.1, 0.0, 0.1]), 
                 metadata={"tenant_id": "power_plant", "clearance": ["engineer"]})
vdb.add_document("doc_2", "Confidential salary scales", np.array([0.8, 0.2, 0.1, 0.0]), 
                 metadata={"tenant_id": "power_plant", "clearance": ["hr_director"]})
vdb.add_document("doc_3", "Retail marketing catalog", np.array([0.1, 0.9, 0.2, 0.1]), 
                 metadata={"tenant_id": "retail_division", "clearance": ["public"]})

# Query with 'engineer' clearance
q_vec = np.array([1.0, 0.0, 0.0, 0.0])
results = vdb.query(q_vec, top_k=5, tenant_id="power_plant", required_role="engineer")
print("Filtered VectorDB Results (Only Authorized Documents):")
for r in results:
    print(f"  [{r['doc_id']}] Score: {r['score']} | {r['text']} | Clearance: {r['metadata']['clearance']}")


---
# ⚡ Section 4: Core Transformer & LLM Architecture From Scratch

```mermaid
flowchart LR
    A[Token IDs] --> B[Embedding + RoPE]
    B --> C[RMSNorm]
    C --> D[Grouped-Query Attention GQA + KV-Cache]
    D --> E[SwiGLU Feed Forward FFN]
    E --> F[Autoregressive Sampler: Temp / Top-K / Top-P]
```


In [ ]:
# 4.1 Byte-Pair Encoding (BPE) Tokenizer from Scratch

class SimpleBPE:
    """
    Byte-Pair Encoding (BPE) algorithm.
    Iteratively counts adjacent token pairs and merges the most frequent pair into a new token ID.
    """
    def __init__(self, vocab_size: int = 50):
        self.target_vocab_size = vocab_size
        self.merges: Dict[Tuple[str, str], str] = {}
        self.vocab = {}
        
    def train(self, corpus: str):
        # Character-level tokens with word boundary marker '</w>'
        words = corpus.split()
        splits = {tuple(list(w) + ['</w>']): 1 for w in words}
        
        while len(self.vocab) < self.target_vocab_size:
            pairs = defaultdict(int)
            for word_tuple, freq in splits.items():
                for i in range(len(word_tuple) - 1):
                    pairs[(word_tuple[i], word_tuple[i+1])] += freq
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            merged_token = "".join(best_pair)
            self.merges[best_pair] = merged_token
            
            # Apply merge
            new_splits = {}
            for word_tuple, freq in splits.items():
                new_tuple = []
                i = 0
                while i < len(word_tuple):
                    if i < len(word_tuple) - 1 and (word_tuple[i], word_tuple[i+1]) == best_pair:
                        new_tuple.append(merged_token)
                        i += 2
                    else:
                        new_tuple.append(word_tuple[i])
                        i += 1
                new_splits[tuple(new_tuple)] = freq
            splits = new_splits
            self.vocab[merged_token] = len(self.vocab)
            if len(self.vocab) >= self.target_vocab_size:
                break
        print(f"✅ BPE Training complete. Learned {len(self.merges)} merge rules.")

bpe = SimpleBPE(vocab_size=15)
bpe.train("low lower lowest newest widest wider low low lowest")
print("Sample BPE Merges:", list(bpe.merges.items())[:5])


In [ ]:
# 4.2 Scaled Dot-Product Attention & Grouped-Query Attention (GQA) with KV-Cache

def scaled_dot_product_attention(
    Q: np.ndarray, 
    K: np.ndarray, 
    V: np.ndarray, 
    mask: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k) + M) V
    """
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.swapaxes(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Stable softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = np.matmul(attn_weights, V)
    return output, attn_weights

class GroupedQueryAttentionWithKVCache:
    """
    Grouped-Query Attention (GQA) as used in Llama-3 (e.g. 8 Query heads sharing 1 KV head).
    Features an autoregressive step-by-step KV-cache to avoid recomputing past tokens.
    """
    def __init__(self, d_model: int = 64, num_q_heads: int = 8, num_kv_heads: int = 2):
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_q_heads
        self.group_size = num_q_heads // num_kv_heads # 4:1 ratio
        
        # KV Cache buffers
        self.cached_k = None # (B, num_kv_heads, cached_len, d_k)
        self.cached_v = None
        
    def reset_cache(self):
        self.cached_k = None
        self.cached_v = None
        
    def forward_step(self, q_step: np.ndarray, k_step: np.ndarray, v_step: np.ndarray) -> np.ndarray:
        """
        q_step: (B, num_q_heads, 1, d_k)
        k_step: (B, num_kv_heads, 1, d_k)
        v_step: (B, num_kv_heads, 1, d_k)
        """
        if self.cached_k is None:
            self.cached_k = k_step
            self.cached_v = v_step
        else:
            self.cached_k = np.concatenate([self.cached_k, k_step], axis=2)
            self.cached_v = np.concatenate([self.cached_v, v_step], axis=2)
            
        # Repeat KV heads to match Q heads (Grouped replication)
        k_rep = np.repeat(self.cached_k, self.group_size, axis=1)
        v_rep = np.repeat(self.cached_v, self.group_size, axis=1)
        
        out, _ = scaled_dot_product_attention(q_step, k_rep, v_rep)
        return out

# Demo GQA with KV-Cache
gqa = GroupedQueryAttentionWithKVCache(d_model=32, num_q_heads=4, num_kv_heads=1)
gqa.reset_cache()

# Simulate decoding 3 tokens autoregressively
for step in range(3):
    q = np.random.randn(1, 4, 1, 8)
    k = np.random.randn(1, 1, 1, 8)
    v = np.random.randn(1, 1, 1, 8)
    out = gqa.forward_step(q, k, v)
    print(f"Step {step+1} output shape: {out.shape}, Cache size: {gqa.cached_k.shape[2]} tokens")


In [ ]:
# 4.3 Rotary Positional Embedding (RoPE)

def apply_rotary_pos_emb(x: np.ndarray, seq_len: int, d_head: int, base: float = 10000.0) -> np.ndarray:
    """
    Rotary Position Embeddings (RoPE) rotates pairs of dimensions by angle m * theta_i:
    [x1, x2] -> [x1 * cos(m*theta) - x2 * sin(m*theta), x1 * sin(m*theta) + x2 * cos(m*theta)]
    """
    dim_indices = np.arange(0, d_head, 2, dtype=np.float32)
    inv_freq = 1.0 / (base ** (dim_indices / d_head))
    positions = np.arange(seq_len, dtype=np.float32)
    
    # Outer product for angles: (seq_len, d_head // 2)
    angles = np.outer(positions, inv_freq)
    sin = np.sin(angles)
    cos = np.cos(angles)
    
    # Apply 2D rotation to consecutive dimension pairs
    x_rot = np.zeros_like(x)
    for pos in range(seq_len):
        for i in range(d_head // 2):
            c = cos[pos, i]
            s = sin[pos, i]
            x1, x2 = x[pos, 2*i], x[pos, 2*i + 1]
            x_rot[pos, 2*i] = x1 * c - x2 * s
            x_rot[pos, 2*i + 1] = x1 * s + x2 * c
    return x_rot

x_test = np.random.randn(4, 8) # 4 tokens, head_dim=8
x_rope = apply_rotary_pos_emb(x_test, seq_len=4, d_head=8)
print("RoPE Embedding Applied! Original vs Rotated norm:", np.linalg.norm(x_test[0]), np.linalg.norm(x_rope[0]))


In [ ]:
# 4.4 Autoregressive Sampling: Temperature, Top-K, Top-P (Nucleus) & Repetition Penalty

def sample_next_token(
    logits: np.ndarray,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 0.9,
    repetition_penalty: float = 1.2,
    generated_tokens: Optional[List[int]] = None
) -> int:
    """
    Full production autoregressive sampling engine.
    1. Apply repetition penalty
    2. Scale logits by Temperature
    3. Apply Top-K filtering
    4. Apply Top-P (Nucleus) cumulative probability cutoff
    5. Sample from resulting multinomial distribution
    """
    logits = logits.copy().astype(np.float64)
    
    # 1. Repetition Penalty
    if generated_tokens and repetition_penalty != 1.0:
        for t in set(generated_tokens):
            if logits[t] > 0:
                logits[t] /= repetition_penalty
            else:
                logits[t] *= repetition_penalty
                
    # 2. Temperature scaling
    if temperature > 0:
        logits = logits / temperature
    else:
        return int(np.argmax(logits))
        
    # 3. Top-K Filter
    if top_k > 0:
        top_k = min(top_k, len(logits))
        indices_to_remove = logits < np.partition(logits, -top_k)[-top_k]
        logits[indices_to_remove] = -np.inf
        
    # Softmax
    exp_logits = np.exp(logits - np.max(logits))
    probs = exp_logits / np.sum(exp_logits)
    
    # 4. Top-P (Nucleus) Filter
    if 0.0 < top_p < 1.0:
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumulative_probs = np.cumsum(sorted_probs)
        
        # Remove tokens with cumulative probability above threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift to keep first token above threshold
        sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1]
        sorted_indices_to_remove[0] = False
        
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        probs[indices_to_remove] = 0.0
        probs = probs / np.sum(probs) # Renormalize
        
    return int(np.random.choice(len(probs), p=probs))

# Demo Sampler
dummy_logits = np.array([2.0, 5.0, 1.0, 0.5, 4.8, -2.0, 3.2])
sampled = sample_next_token(dummy_logits, temperature=0.7, top_k=3, top_p=0.85)
print(f"Sampled Token ID from distribution: {sampled}")


---
# 📦 Section 5: Quantization & Parameter-Efficient Fine-Tuning (PEFT)

INT8 Quantization reduces model VRAM by 50% vs FP16, and LoRA allows fine-tuning massive LLMs by training $<0.1\%$ low-rank parameters.


In [ ]:
# 5.1 Symmetric & Asymmetric INT8 Quantization Engine

def quantize_symmetric_int8(weights: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Symmetric INT8: Maps [-max_val, max_val] to [-127, 127] with ZeroPoint = 0.
    Scale S = max(|W|) / 127
    """
    max_val = np.max(np.abs(weights))
    scale = float(max_val / 127.0) if max_val > 0 else 1.0
    q_weights = np.clip(np.round(weights / scale), -127, 127).astype(np.int8)
    return q_weights, scale

def dequantize_symmetric_int8(q_weights: np.ndarray, scale: float) -> np.ndarray:
    return q_weights.astype(np.float32) * scale

# Benchmark Quantization
fp32_weights = np.random.randn(1000, 1000).astype(np.float32)
q8_weights, scale = quantize_symmetric_int8(fp32_weights)
reconstructed = dequantize_symmetric_int8(q8_weights, scale)

mae = np.mean(np.abs(fp32_weights - reconstructed))
print(f"FP32 Size: {fp32_weights.nbytes / 1024:.1f} KB -> INT8 Size: {q8_weights.nbytes / 1024:.1f} KB (4x Compression)")
print(f"Quantization Mean Absolute Reconstruction Error: {mae:.6f}")


In [ ]:
# 5.2 LoRA (Low-Rank Adaptation) Layer from Scratch

class LoRALinear:
    """
    LoRA Layer: W = W_0 + (alpha / r) * (B @ A)
    - W_0 is frozen (original weights)
    - A is initialized with Gaussian N(0, 1/r)
    - B is initialized with Zeros (so initial Delta W = 0)
    """
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 8.0):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = alpha / rank
        
        # Frozen base weight
        self.W0 = np.random.randn(out_features, in_features).astype(np.float32) * 0.02
        
        # Trainable low-rank adapters
        self.A = np.random.randn(rank, in_features).astype(np.float32) * (1.0 / rank)
        self.B = np.zeros((out_features, rank), dtype=np.float32)
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        # Base forward + LoRA delta
        base_out = np.dot(x, self.W0.T)
        lora_out = np.dot(np.dot(x, self.A.T), self.B.T) * self.scaling
        return base_out + lora_out

lora = LoRALinear(in_features=128, out_features=128, rank=4, alpha=8.0)
x_in = np.random.randn(2, 128)
print("LoRA Forward Output Shape:", lora.forward(x_in).shape)


---
# 🛡️ Section 6: Production Agentic Tooling & System Infrastructure

High-throughput enterprise GenAI architectures require:
1. **Semantic Model Routers** (60% SLM routing cuts platform cost by >70%)
2. **Air-Gapped Tool Calling Gatekeepers** (Prevents unauthorized writes/actions)
3. **Context Budget Compressors** (Prevents context window overflow)


In [ ]:
# 6.1 Semantic Model Router (SLM vs Frontier Cost Optimizer)

class SemanticModelRouter:
    """
    Routes incoming prompts to appropriate model tiers based on intent classification & complexity.
    - Low Complexity (Summaries, translations, simple Q&A) -> 2B SLM ($0.00005/req)
    - High Complexity (Multi-step reasoning, coding, contract analysis) -> 70B Frontier ($0.004/req)
    """
    def __init__(self):
        self.slm_intents = {"summarization", "grammar_check", "translation", "faq"}
        self.frontier_intents = {"code_generation", "mathematical_proof", "contract_risk", "system_architecture"}
        
    def route_request(self, prompt: str, detected_intent: str) -> Dict[str, Any]:
        word_count = len(prompt.split())
        
        if detected_intent in self.slm_intents and word_count < 250:
            return {
                "target_model": "Llama-3.2-3B-Instruct",
                "tier": "SLM_EDGE",
                "estimated_cost_per_req": 0.00005,
                "reason": "Standard task within SLM capacity"
            }
        else:
            return {
                "target_model": "Llama-3.3-70B-Instruct-FP8",
                "tier": "FRONTIER_REASONING",
                "estimated_cost_per_req": 0.00080,
                "reason": "High complexity / long-context intent requiring frontier reasoning"
            }

router = SemanticModelRouter()
print("Route 1:", router.route_request("Summarize this 2-sentence paragraph", "summarization"))
print("Route 2:", router.route_request("Review this indemnification liability clause", "contract_risk"))


In [ ]:
# 6.2 Air-Gapped Tool Calling & RBAC Gatekeeper

class ToolExecutionGatekeeper:
    """
    Validates LLM tool invocations against JSON Schema, RBAC permissions, and write gates.
    """
    def __init__(self):
        self.read_tools = {"get_machine_telemetry", "query_inventory_db"}
        self.write_tools = {"trigger_valve_cutoff", "update_payroll_record"}
        
    def execute_tool(self, tool_name: str, arguments: Dict[str, Any], user_roles: List[str]) -> Dict[str, Any]:
        # 1. Check if tool exists
        all_tools = self.read_tools | self.write_tools
        if tool_name not in all_tools:
            return {"status": "ERROR", "message": f"Tool '{tool_name}' is not registered."}
            
        # 2. Write Gate & RBAC Verification
        if tool_name in self.write_tools:
            if "plant_supervisor" not in user_roles and "admin" not in user_roles:
                return {
                    "status": "APPROVAL_REQUIRED",
                    "message": f"Tool '{tool_name}' performs destructive state change. Awaiting supervisor sign-off.",
                    "held_payload": arguments
                }
                
        # 3. Simulate execution
        return {
            "status": "SUCCESS",
            "executed_tool": tool_name,
            "result": f"Successfully executed with args: {arguments}"
        }

gate = ToolExecutionGatekeeper()
# Unauthorized write attempt
res1 = gate.execute_tool("trigger_valve_cutoff", {"valve_id": "V-401"}, user_roles=["operator"])
print("Write Gate Intercept (Operator):", res1)

# Authorized read attempt
res2 = gate.execute_tool("get_machine_telemetry", {"turbine_id": "T-10"}, user_roles=["operator"])
print("Read Execution (Operator):", res2)


---
# 📊 Section 7: The Complete GenAI & RAG Evaluation Suite

### 1. The RAG Triad
- **Faithfulness / Groundedness**: Are all claims in the generated response grounded in the context?
- **Context Relevance / Precision**: Does the context contain the answer without noisy distractor tokens?
- **Answer Relevance**: Does the answer address the question?

### 2. Traditional LLM Metrics
- **Perplexity (PPL)**: Fluency metric $PPL = \exp(\mathcal{L}_{\text{CE}})$.
- **ROUGE-1, ROUGE-2, ROUGE-L**: Unigram, Bigram, and Longest Common Subsequence (LCS) overlap.
- **BLEU-1 to BLEU-4 with Brevity Penalty**: N-gram precision.
- **Exact Match (EM) & Token F1**: Precision/Recall of answer tokens.


In [ ]:
# 7.1 The RAG Triad Evaluator from Scratch

class RAGTriadEvaluator:
    """
    Evaluates RAG retrieval and generation quality without external APIs.
    """
    @staticmethod
    def context_relevance(query: str, context: str) -> float:
        """Measures term overlap and key entity presence of query in context."""
        q_tokens = set(re.findall(r'\w+', query.lower()))
        c_tokens = set(re.findall(r'\w+', context.lower()))
        if not q_tokens:
            return 1.0
        return len(q_tokens.intersection(c_tokens)) / len(q_tokens)
        
    @staticmethod
    def faithfulness(context: str, generated_answer: str) -> float:
        """
        Measures the fraction of factual n-grams in the answer that are supported by the context.
        """
        ans_sentences = [s.strip() for s in generated_answer.split('.') if s.strip()]
        if not ans_sentences:
            return 1.0
            
        grounded_count = 0
        c_lower = context.lower()
        for sentence in ans_sentences:
            s_tokens = set(re.findall(r'\w+', sentence.lower()))
            # If 60% of significant tokens exist in context, mark as grounded
            if s_tokens:
                overlap = len(s_tokens.intersection(set(re.findall(r'\w+', c_lower)))) / len(s_tokens)
                if overlap >= 0.60:
                    grounded_count += 1
                    
        return grounded_count / len(ans_sentences)
        
    @staticmethod
    def answer_relevance(query: str, generated_answer: str) -> float:
        """Measures overlap between query and generated response."""
        q_tokens = set(re.findall(r'\w+', query.lower()))
        a_tokens = set(re.findall(r'\w+', generated_answer.lower()))
        if not q_tokens:
            return 1.0
        return len(q_tokens.intersection(a_tokens)) / len(q_tokens)

# Run RAG Triad Test
query_sample = "What is the recommended maintenance cycle for the gas turbine?"
context_sample = "The gas turbine model GT-9000 requires full inspection every 10,000 operational hours."
grounded_answer = "The GT-9000 gas turbine requires complete inspection every 10,000 hours."
hallucinated_answer = "The turbine must be replaced with nuclear steam valves every 2 days."

print("Grounded Answer Evaluation:")
print(f"  Context Relevance: {RAGTriadEvaluator.context_relevance(query_sample, context_sample):.2f}")
print(f"  Faithfulness:      {RAGTriadEvaluator.faithfulness(context_sample, grounded_answer):.2f}")
print(f"  Answer Relevance:  {RAGTriadEvaluator.answer_relevance(query_sample, grounded_answer):.2f}")

print("\nHallucinated Answer Evaluation:")
print(f"  Faithfulness:      {RAGTriadEvaluator.faithfulness(context_sample, hallucinated_answer):.2f}")


In [ ]:
# 7.2 Perplexity, ROUGE-L, BLEU-4, and Token F1-Score

def compute_perplexity(cross_entropy_losses: List[float]) -> float:
    """Perplexity = exp(mean(CrossEntropyLoss))"""
    return float(np.exp(np.mean(cross_entropy_losses)))

def compute_token_f1(prediction: str, ground_truth: str) -> Tuple[float, float, float]:
    """Computes Token-level Precision, Recall, and F1-score."""
    pred_tokens = re.findall(r'\w+', prediction.lower())
    gold_tokens = re.findall(r'\w+', ground_truth.lower())
    
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens), int(pred_tokens == gold_tokens), int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0, 0.0, 0.0
        
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return precision, recall, f1

def compute_rouge_l(candidate: str, reference: str) -> float:
    """Computes ROUGE-L using Longest Common Subsequence (LCS)."""
    cand_tokens = re.findall(r'\w+', candidate.lower())
    ref_tokens = re.findall(r'\w+', reference.lower())
    
    m, n = len(cand_tokens), len(ref_tokens)
    if m == 0 or n == 0:
        return 0.0
        
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if cand_tokens[i-1] == ref_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
                
    lcs_len = dp[m][n]
    precision = lcs_len / m
    recall = lcs_len / n
    if precision + recall == 0:
        return 0.0
    return (2 * precision * recall) / (precision + recall)

def compute_bleu_4(candidate: str, reference: str) -> float:
    """Computes BLEU-4 with Brevity Penalty from scratch."""
    cand_tokens = re.findall(r'\w+', candidate.lower())
    ref_tokens = re.findall(r'\w+', reference.lower())
    
    c_len = len(cand_tokens)
    r_len = len(ref_tokens)
    if c_len == 0:
        return 0.0
        
    # Brevity Penalty
    bp = 1.0 if c_len > r_len else math.exp(1.0 - r_len / c_len)
    
    precisions = []
    for n in range(1, 5):
        cand_ngrams = [tuple(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)]
        ref_ngrams = [tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)]
        
        if not cand_ngrams:
            precisions.append(1e-9)
            continue
            
        cand_counts = Counter(cand_ngrams)
        ref_counts = Counter(ref_ngrams)
        overlap = sum((cand_counts & ref_counts).values())
        precisions.append(max(overlap / len(cand_ngrams), 1e-9))
        
    log_avg = sum(math.log(p) for p in precisions) / 4.0
    return bp * math.exp(log_avg)

# Benchmark Metrics
pred = "The turbine runs smoothly at high rotational velocity."
ref = "The turbine operates smoothly at high rotational speed."

p, r, f1 = compute_token_f1(pred, ref)
rouge_l = compute_rouge_l(pred, ref)
bleu_4 = compute_bleu_4(pred, ref)
ppl = compute_perplexity([1.2, 1.4, 1.1, 1.3])

print(f"📊 Evaluation Benchmark Results:")
print(f"  Token F1-Score: {f1:.4f} (Precision: {p:.4f}, Recall: {r:.4f})")
print(f"  ROUGE-L Score:  {rouge_l:.4f}")
print(f"  BLEU-4 Score:   {bleu_4:.4f}")
print(f"  Perplexity:     {ppl:.4f}")


---
# Section 8 — LLM-as-Judge

## 8.1 Why LLM-as-Judge?

Lexical metrics (ROUGE, BLEU) correlate poorly with human judgement for open-ended generation. MAANG teams use a **stronger model (GPT-4, Gemini-1.5-Pro) as an automated judge** — called **G-Eval** (Liu et al., 2023) — to score outputs on semantic quality dimensions.

| Dimension | Definition |
| :--- | :--- |
| **Coherence** | Is the response logically structured and fluent? |
| **Faithfulness** | Does the response only contain facts from the context? |
| **Relevance** | Does the response directly answer the question? |
| **Fluency** | Is the language natural and grammatically correct? |

## 8.2 Known Biases in LLM Judges

| Bias | Description | Mitigation |
| :--- | :--- | :--- |
| **Position Bias** | Judge prefers responses shown first | Randomise A/B order |
| **Verbosity Bias** | Judge prefers longer responses | Penalise length in prompt |
| **Self-Enhancement** | Model prefers its own family outputs | Use a different-family judge |
| **Sycophancy** | Judge agrees with prompt assertions | Chain-of-thought + explicit rubric |


In [ ]:
# -- 8.3  LLM-as-Judge (G-Eval Style) --------------------------------------
# Production: call GPT-4/Gemini with a scoring prompt. Here: mock scores.

import json
import numpy as np
from typing import List, Dict, Optional
np.random.seed(42)


def mock_llm_judge(prompt: str, context: str, response: str, quality: str = 'high') -> Dict:
    """Simulate LLM judge scores. quality in {high, medium, low}."""
    rng  = np.random.RandomState(abs(hash(response)) % (2**31))
    base = {'high': 4.5, 'medium': 3.0, 'low': 1.8}[quality]
    dims = ['coherence', 'faithfulness', 'relevance', 'fluency']
    scores = {d: int(np.clip(rng.normal(base, 0.4), 1, 5)) for d in dims}
    scores['rationale'] = f'Response quality is {quality}.'
    scores['composite'] = round(np.mean([scores[d] for d in dims]), 2)
    return scores


eval_cases = [
    {
        'prompt':     'What is LoRA fine-tuning?',
        'context':    'LoRA injects trainable low-rank matrices into frozen LLM weights.',
        'response_a': 'LoRA injects trainable low-rank matrices A and B into frozen W0. Only A and B are trained.',
        'response_b': 'LoRA is a technique for making LLMs faster by quantization.',
    },
    {
        'prompt':     'Explain KV-Cache in transformers.',
        'context':    'KV-Cache stores key-value tensors to avoid recomputation during autoregressive decoding.',
        'response_a': 'KV-Cache saves key and value tensors from past tokens so each step only attends to cached states.',
        'response_b': 'KV-Cache is GPU memory used to store model weights during inference.',
    },
    {
        'prompt':     'What is Reciprocal Rank Fusion?',
        'context':    'RRF combines ranked lists: score = sum(1/(k + rank_i)), k=60.',
        'response_a': 'RRF fuses ranked lists using 1/(k+rank) per system summed across systems. k=60 dampens outliers.',
        'response_b': 'RRF is a ML model that re-ranks results using neural embeddings.',
    },
]

print('Evaluating with LLM-as-Judge (G-Eval)...')
print('=' * 65)
for i, case in enumerate(eval_cases):
    sa = mock_llm_judge(case['prompt'], case['context'], case['response_a'], 'high')
    sb = mock_llm_judge(case['prompt'], case['context'], case['response_b'], 'low')
    winner = 'A' if sa['composite'] >= sb['composite'] else 'B'
    print(f"Case {i+1}: {case['prompt']}")
    print(f"  A={sa['composite']}  B={sb['composite']}  -> Winner: {winner}")
    for d in ['coherence', 'faithfulness', 'relevance', 'fluency']:
        print(f"    {d:<14}: A={sa[d]}  B={sb[d]}")
    print()


In [ ]:
# -- 8.4  Correlation: LLM Judge vs ROUGE-L ----------------------------------
import matplotlib.pyplot as plt
plt.style.use('dark_background')

np.random.seed(0)
n = 40
quality      = np.random.uniform(1, 5, n)
llm_scores   = np.clip(quality + np.random.normal(0, 0.3, n), 1, 5)
rouge_scores = np.clip(quality * 0.12 + np.random.normal(0.3, 0.15, n), 0, 1)

r_llm   = np.corrcoef(quality, llm_scores)[0, 1]
r_rouge = np.corrcoef(quality, rouge_scores)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LLM-as-Judge vs ROUGE-L Correlation with True Quality', fontsize=13, color='#00E5FF')

for ax, scores, label, color in [
    (axes[0], llm_scores,   f'LLM Judge  r={r_llm:.2f}',   '#00FF88'),
    (axes[1], rouge_scores, f'ROUGE-L    r={r_rouge:.2f}', '#FF4B6E'),
]:
    ax.scatter(quality, scores, color=color, s=60, alpha=0.8)
    m, b = np.polyfit(quality, scores, 1)
    ax.plot(quality, m * quality + b, color='white', lw=1.5, ls='--')
    ax.set_title(label, color='white')
    ax.set_xlabel('True Quality (1-5)')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()
print(f'LLM Judge r = {r_llm:.2f}  |  ROUGE-L r = {r_rouge:.2f}')
print(f'LLM Judge correlates {r_llm/r_rouge:.1f}x better with human quality')


---
# Section 9 — Structured Output & Constrained Decoding

## 9.1 The Problem

LLMs generate free-form text. In production you need **guaranteed valid structure** — JSON for APIs, SQL for databases, typed models for downstream systems.

## 9.2 Four Approaches

| Approach | Mechanism | Reliability | Overhead |
| :--- | :--- | :--- | :--- |
| **Post-hoc parsing** | Generate → `json.loads()` → retry on fail | Low | None |
| **JSON-mode / Prompt** | Instruct model to return JSON | Medium | None |
| **Logit masking** | Mask invalid tokens per step | High | +5–15% |
| **Grammar sampling (FSM)** | FSM enforces grammar per token | Highest | +10–20% |

## 9.3 Production Libraries
- **`outlines`**: FSM grammar sampling, any HuggingFace model
- **`instructor`**: Pydantic JSON mode for OpenAI / Anthropic / Gemini
- **vLLM `guided_decoding`**: Built-in grammar sampling engine


In [ ]:
# -- 9.4  Pydantic-Style Schema Validation -----------------------------------
import json
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class RiskClause:
    clause_id: str
    risk_type: str
    severity: str
    flagged_text: str
    confidence: float
    recommended_action: str


@dataclass
class ContractRiskReport:
    contract_id: str
    analysed_by: str
    overall_risk_level: str
    clauses: List[RiskClause] = field(default_factory=list)
    summary: str = ''


RAW = json.dumps({
    'contract_id': 'MSA-2026-001',
    'analysed_by': 'LLM-LoRA-Legal-DeBERTa-v3',
    'overall_risk_level': 'high',
    'summary': 'Broad indemnification clause with uncapped liability exposure.',
    'clauses': [
        {'clause_id': 'C-04', 'risk_type': 'indemnification', 'severity': 'critical',
         'flagged_text': 'Vendor shall indemnify Client against all claims of any nature.',
         'confidence': 0.97, 'recommended_action': 'Negotiate cap to 12-month contract value.'},
        {'clause_id': 'C-09', 'risk_type': 'governing_law', 'severity': 'medium',
         'flagged_text': 'Governed by the laws of the State of Delaware.',
         'confidence': 0.88, 'recommended_action': 'Confirm local counsel review.'},
    ]
}, indent=2)


def parse_and_validate(raw: str) -> Optional[ContractRiskReport]:
    VALID = {'low', 'medium', 'high', 'critical'}
    try:
        d = json.loads(raw)
        clauses = [RiskClause(**c) for c in d.pop('clauses', [])]
        report  = ContractRiskReport(**d, clauses=clauses)
        for c in report.clauses:
            assert c.severity in VALID,          f'Invalid severity: {c.severity}'
            assert 0.0 <= c.confidence <= 1.0,   f'Confidence OOB: {c.confidence}'
        return report
    except Exception as e:
        print(f'Validation error: {e}')
        return None


report = parse_and_validate(RAW)
if report:
    print(f'\u2705 Structured output validated!')
    print(f'   Contract:      {report.contract_id}')
    print(f'   Risk Level:    {report.overall_risk_level.upper()}')
    print(f'   Clauses found: {len(report.clauses)}')
    for c in report.clauses:
        print(f'   [{c.severity.upper():<8}] {c.risk_type} (conf={c.confidence}) -> {c.recommended_action[:55]}...')


In [ ]:
# -- 9.5  Logit Masking: Token-Level Constrained Decoding -------------------
import numpy as np

VOCAB     = ['{', '}', ':', ',', '"', 'risk', 'id', 'high', 'low', '0', '1', 'null', 'true']
VOCAB_IDX = {tok: i for i, tok in enumerate(VOCAB)}


def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()


def mask_logits(logits, allowed):
    m = np.full_like(logits, -np.inf)
    for tok in allowed:
        if tok in VOCAB_IDX:
            m[VOCAB_IDX[tok]] = logits[VOCAB_IDX[tok]]
    return m


np.random.seed(1)
raw = np.random.randn(len(VOCAB))
msk = mask_logits(raw, ['{'])    # Only '{' valid at position 0 of JSON

p_raw = softmax(raw)
p_msk = softmax(np.where(np.isneginf(msk), -1e9, msk))

print('Logit Masking at JSON position 0 (only "{" is valid):')
print(f'{"Token":<10} {"Raw Prob":>10} {"Masked":>10} {"Valid":>6}')
print('-' * 42)
for tok, pr, pm in zip(VOCAB, p_raw, p_msk):
    mark = '\u2705' if tok == '{' else '\u274c'
    print(f'{tok:<10} {pr:>10.4f} {pm:>10.4f} {mark:>6}')
print('\n-> After masking: "{" gets 100% probability -- valid JSON guaranteed.')


In [ ]:
# -- 9.6  FSM Grammar Sampling (outlines-style concept) ---------------------
# A Finite-State Machine defines legal token sequences.
# At each decoding step, tokens with no valid FSM transition are masked.
from dataclasses import dataclass, field
from typing import Dict as TDict, Optional


@dataclass
class FSMState:
    name: str
    is_terminal: bool = False
    transitions: TDict[str, str] = field(default_factory=dict)

    def allowed(self):   return set(self.transitions)
    def step(self, tok): return self.transitions.get(tok)


# Minimal FSM for: { "key": "value" }
STATES = {
    'START':          FSMState('START',          transitions={'{':     'OPEN_BRACE'}),
    'OPEN_BRACE':     FSMState('OPEN_BRACE',     transitions={'"':     'KEY_START'}),
    'KEY_START':      FSMState('KEY_START',       transitions={'risk': 'KEY_END', 'id': 'KEY_END'}),
    'KEY_END':        FSMState('KEY_END',         transitions={'"':     'COLON_WAIT'}),
    'COLON_WAIT':     FSMState('COLON_WAIT',      transitions={':':     'VALUE_START'}),
    'VALUE_START':    FSMState('VALUE_START',      transitions={'"':     'VALUE_TEXT'}),
    'VALUE_TEXT':     FSMState('VALUE_TEXT',       transitions={'high': 'VALUE_END', 'low': 'VALUE_END'}),
    'VALUE_END':      FSMState('VALUE_END',        transitions={'"':     'CLOSE_OR_COMMA'}),
    'CLOSE_OR_COMMA': FSMState('CLOSE_OR_COMMA',   transitions={'}': 'DONE', ',': 'OPEN_BRACE'}),
    'DONE':           FSMState('DONE',             is_terminal=True),
}


def simulate_fsm(tokens):
    state = 'START'
    print(f'{"Step":<5} {"Token":<10} {"From":<16} {"To":<16} {"Status"}')
    print('-' * 58)
    for i, tok in enumerate(tokens):
        nxt = STATES[state].step(tok)
        ok  = '\u2705 OK' if nxt else '\u274c BLOCKED'
        print(f'{i:<5} {repr(tok):<10} {state:<16} {str(nxt or "--"):<16} {ok}')
        if not nxt:
            print('  Token masked -- FSM blocks this generation path.')
            return
        state = nxt
    if STATES[state].is_terminal:
        print('\n\u2705 Terminal state -- valid grammar sequence!')


print('=== Valid sequence: { "risk": "high" } ===')
simulate_fsm(['{', '"', 'risk', '"', ':', '"', 'high', '"', '}'])
print()
print('=== Invalid sequence (missing ":") ===')
simulate_fsm(['{', '"', 'risk', '"', 'high'])  # 'high' after KEY_END is illegal


---
## 🏆 Summary & Takeaways

You have now built every core algorithm tested in MAANG GenAI interviews from scratch:
- **Vector Metric Foundations**: Cosine, L2, Angular Distance.
- **Search & Indexing**: BM25, IVFFlat K-Means, HNSW Graph, RRF.
- **Production VectorDB**: Multi-Tenant ACL & Top-$K$ semantic querying.
- **Transformer Internals**: BPE Tokenizer, Scaled Dot-Product MHA, GQA with KV-Cache, RoPE.
- **PEFT & Quantization**: Symmetric INT8 Quantizer, LoRA Layer.
- **Infrastructure & Routing**: Cost-optimizing Model Router, Air-gapped write gatekeeper.
- **Comprehensive Evaluation**: RAG Triad, Perplexity, ROUGE-L, BLEU-4, Token F1.
- **LLM-as-Judge (G-Eval)**: Coherence / Faithfulness / Relevance / Fluency scoring, bias taxonomy, correlation analysis.
- **Structured Output**: Pydantic JSON schema validation, logit masking, FSM-based grammar sampling.
